# Feature selection summary

In [1]:
import glob
import os
import time
import warnings
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
%matplotlib inline

from sgimc.utils import save, load

import pandas as pd

In [2]:
PATH_ROOT = "/Users/sijianfan/projects/BiSSGL"
PATH_DATA = os.path.join(PATH_ROOT, "datasets/realAnalysis/cdataset/cv_data")

PATH_OUTPUT = os.path.join(PATH_ROOT, "outputs/results/realAnalysis/cdataset/noisy")

if not os.path.isdir(PATH_OUTPUT):
    os.makedirs(PATH_OUTPUT)

PATH_ARCHIVE = os.path.join(PATH_OUTPUT, "archived")
if not os.path.isdir(PATH_ARCHIVE):
    os.makedirs(PATH_ARCHIVE)

In [3]:
filename_results = []
filename_results.append(os.path.join(PATH_OUTPUT, "results_bissgl_comb.gz"))

In [4]:
results = []
for fn in filename_results:
    results.append(load(fn))

In [5]:
results

[[{'combo_idx': 0,
   'train_size': 0.1,
   'xi': 1,
   'eta': 1e-07,
   'tilde_lambda0': 100,
   'tilde_lambda1': 1,
   'lambda0': 100,
   'lambda1': 1,
   'K': 200,
   'cv': 0,
   'val_score': [0.009655692026810294,
    0.5547658904042145,
    0.03764705882352941,
    0.9240200631618057,
    0.21052631578947367,
    0.9290926099158092,
    0.020671834625323],
   'val_d1': 24,
   'val_d2': 132,
   'val_d1_idx': array([ 691,  738,  805, 1013, 1307, 1383, 1464, 1506, 1550, 1583, 1676,
          1746, 1747, 1753, 1758, 2251, 2252, 2305, 2424, 2426, 2430, 2438,
          2444, 2474]),
   'val_d2_idx': array([   5,    8,   22,   31,   45,   60,   64,   66,   83,   85,   99,
           106,  120,  134,  143,  159,  191,  209,  210,  211,  212,  222,
           227,  228,  232,  243,  250,  258,  260,  290,  294,  299,  300,
           309,  314,  315,  316,  320,  326,  327,  335,  337,  344,  345,
           348,  349,  350,  355,  356,  357,  358,  359,  360,  361,  362,
           365,  

In [6]:
METRIC_INDEX = {
    "aupr": 0,
    "auc": 1,
    "f1": 2,
    "acc": 3,
    "recall": 4,
    "specificity": 5,
    "precision": 6,
}

from collections import defaultdict
import numpy as np


def get_final_results(results, param="train_size", hyp_params=None, target="auc"):

    metric_idx = METRIC_INDEX[target]

    if hyp_params is None:
        hyp_params = []

    # =====================================================
    # 1️⃣ Collapse CV folds → group by full model identity
    # =====================================================
    by_model = defaultdict(list)

    for r in results:
        key = tuple([r[param]] + [r[h] for h in hyp_params])
        by_model[key].append(r)

    model_summary = []

    for key, group in by_model.items():
        p_val = key[0]
        hyp_vals = key[1:]  # all hyperparameters

        val_vals = [g["val_score"][metric_idx] for g in group]
        val_mean = np.nanmean(val_vals)

        test_val = group[0]["test_score"][metric_idx]

        model_summary.append(
            {
                "param": p_val,
                "hyp_vals": hyp_vals,
                "val_mean": val_mean,
                "test_metric": test_val,
                "model_dict": group[0],  # store original dict
            }
        )

    # =====================================================
    # 2️⃣ Select best model per param
    # =====================================================
    by_param = defaultdict(list)

    for row in model_summary:
        by_param[row["param"]].append(row)

    results_final = []
    best_models = {}

    for p_val, rows in by_param.items():

        val_scores = np.array([r["val_mean"] for r in rows])
        best_idx = np.nanargmax(val_scores)
        best = rows[best_idx]

        # results_final.append([p_val, best["test_metric"]])
        results_final.append([p_val, best["val_mean"]])

        # store best model info
        best_models[p_val] = {
            "val_mean": best["val_mean"],
            "test_metric": best["test_metric"],
            **{h: v for h, v in zip(hyp_params, best["hyp_vals"])},
        }
        # print out the best model info
        print(f"\nBest model for {param} = {p_val}")
        print("Validation mean:", best["val_mean"])
        print("Test metric:", best["test_metric"])
        print("Full model dict:")
        print(best["model_dict"])

    results_final = np.array(results_final)

    # =====================================================
    # 3️⃣ Sort for plotting
    # =====================================================
    order = np.argsort(results_final[:, 0])
    results_final = results_final[order]

    return results_final.T

In [7]:
results_bissgl = get_final_results(
    results[0],
    param="train_size",
    hyp_params=["combo_idx"],
    target="auc",
)


Best model for train_size = 0.1
Validation mean: 0.7021345139998656
Test metric: 0.6654171107748674
Full model dict:
{'combo_idx': 5, 'train_size': 0.1, 'xi': 20, 'eta': 1e-07, 'tilde_lambda0': 100, 'tilde_lambda1': 1, 'lambda0': 100, 'lambda1': 1, 'K': 200, 'cv': 0, 'val_score': [0.06007634191798997, 0.6750644329896908, 0.13245033112582782, 0.9756641278097715, 0.20833333333333334, 0.9825679475164011, 0.0970873786407767], 'val_d1': 378, 'val_d2': 308, 'val_d1_idx': array([  70,  143,  265,  658,  659,  661,  662,  667,  671,  673,  691,
        694,  703,  717,  722,  724,  729,  732,  738,  742,  748,  760,
        763,  772,  776,  777,  779,  780,  786,  788,  796,  798,  800,
        805,  809,  823,  833,  853,  855,  870,  874,  877,  880,  889,
        891,  893,  904,  908,  909,  915,  941,  942,  943,  952,  959,
        967,  991,  994,  995,  998,  999, 1002, 1013, 1017, 1018, 1023,
       1035, 1038, 1043, 1044, 1046, 1049, 1055, 1058, 1064, 1066, 1067,
       1080, 1083,

In [ ]:
from collections import Counter


def count_cv_frequency(results, key="val_d1_idx"):
    counter = Counter()

    for r in results[0]:  # only CV list
        counter.update(r[key].tolist())

    return counter

In [9]:
d1_freq = count_cv_frequency(results, key="val_d1_idx")
d2_freq = count_cv_frequency(results, key="val_d2_idx")

print(d1_freq)
print(d2_freq)

Counter({691: 400, 738: 400, 805: 400, 1013: 400, 1307: 400, 1383: 400, 1464: 400, 1506: 400, 1583: 400, 1746: 400, 1747: 400, 1753: 400, 1758: 400, 2251: 400, 2252: 400, 2305: 400, 2424: 400, 2426: 400, 2430: 400, 2474: 400, 2438: 399, 2444: 398, 1550: 397, 1749: 396, 1676: 395, 786: 395, 1035: 395, 1316: 395, 662: 394, 1592: 394, 2428: 393, 1681: 393, 2253: 392, 659: 390, 2257: 390, 1352: 388, 2202: 387, 2425: 386, 2439: 384, 722: 380, 2434: 380, 2262: 379, 1767: 377, 2427: 377, 2267: 375, 833: 375, 2432: 374, 2254: 373, 1769: 373, 1770: 372, 2433: 371, 2443: 370, 1046: 368, 1538: 368, 2459: 367, 1765: 367, 1018: 366, 2270: 366, 2442: 366, 2471: 366, 748: 366, 941: 365, 2440: 362, 1313: 361, 1750: 360, 1788: 360, 1913: 359, 1473: 358, 1921: 358, 2431: 358, 673: 358, 1778: 357, 1790: 357, 1988: 355, 2481: 355, 2435: 355, 1532: 354, 1761: 354, 1772: 354, 1332: 352, 1206: 352, 1387: 351, 1814: 351, 952: 351, 1781: 350, 1085: 349, 2441: 349, 2255: 349, 1754: 348, 1654: 348, 2263: 348, 69

In [10]:
PATH_TO_EXP = "/Users/sijianfan/projects/BiSSGL/datasets/realAnalysis/cdataset"
PATH_DATA = os.path.join(PATH_TO_EXP, "cv_data")
if not os.path.isdir(PATH_DATA):
    os.mkdir(PATH_DATA)

In [11]:
import pandas as pd

df_Y = pd.read_table(os.path.join(PATH_TO_EXP, "c_admat_dgc.txt"), index_col=0)
print(df_Y.shape)

(409, 658)


In [12]:
drugs_features = pd.read_csv(
    os.path.join(PATH_TO_EXP, "drugs_features.csv"), index_col=0
)
drugs_features.shape

(658, 1888)

In [13]:
drugs_features.head()

,ECFP_0,ECFP_1,ECFP_2,ECFP_3,ECFP_4,ECFP_5,ECFP_6,ECFP_7,ECFP_8,ECFP_9,...,GO_CC_GO:0014704,GO_CC_GO:0098552,GO_CC_GO:0031901,GO_CC_GO:0031966,GO_CC_GO:0071944,GO_CC_GO:0016600,GO_CC_GO:0045177,GO_CC_GO:0060076,GO_CC_GO:0034707,GO_CC_GO:1904813
DB00014,0,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DB00035,0,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DB00091,0,1,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
DB00104,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DB00115,1,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [14]:
diseases_features = pd.read_csv(
    os.path.join(PATH_TO_EXP, "diseases_features.csv"), index_col=0
)
diseases_features.shape

(409, 797)

In [15]:
diseases_features.columns

Index(['HPO_HP:0000006', 'HPO_HP:0000707', 'HPO_HP:0012638', 'HPO_HP:0012823',
       'HPO_HP:0031797', 'HPO_HP:0003674', 'HPO_HP:0033127', 'HPO_HP:0001939',
       'HPO_HP:0011446', 'HPO_HP:0000007',
       ...
       'HPO_HP:0100273', 'HPO_HP:0100315', 'HPO_HP:0100627', 'HPO_HP:0100707',
       'HPO_HP:0100738', 'HPO_HP:0100834', 'HPO_HP:0100836', 'HPO_HP:0100872',
       'HPO_HP:0200008', 'HPO_HP:5200230'],
      dtype='object', length=797)

In [22]:
d1_freq.keys()
d1_keys = [int(k) for k in d1_freq.keys() if int(k) <= 1887]
d2_keys = [int(k) for k in d2_freq.keys() if int(k) <= 796]

In [33]:
print(drugs_features.columns[d1_keys][:10])
print(diseases_features.columns[d2_keys][:10])

Index(['ECFP_692', 'ECFP_739', 'ECFP_806', 'ECFP_1014', 'GO_BP_GO:0007195',
       'GO_BP_GO:0051583', 'GO_BP_GO:0045665', 'GO_BP_GO:0042908',
       'GO_BP_GO:0050877', 'GO_BP_GO:0035095'],
      dtype='object')
Index(['HPO_HP:0003674', 'HPO_HP:0011446', 'HPO_HP:0410280', 'HPO_HP:0100022',
       'HPO_HP:0040064', 'HPO_HP:0002012', 'HPO_HP:0002060', 'HPO_HP:0011354',
       'HPO_HP:0002817', 'HPO_HP:0012649'],
      dtype='object')


In [34]:
import numpy as np


def get_metrics(real_score, predict_score):
    sorted_predict_score = np.array(
        sorted(list(set(np.array(predict_score).flatten())))
    )
    sorted_predict_score_num = len(sorted_predict_score)
    thresholds = sorted_predict_score[
        np.int32(sorted_predict_score_num * np.arange(1, 1000) / 1000)
    ]
    thresholds = np.mat(thresholds)
    thresholds_num = thresholds.shape[1]

    predict_score_matrix = np.tile(predict_score, (thresholds_num, 1))
    negative_index = np.where(predict_score_matrix < thresholds.T)
    positive_index = np.where(predict_score_matrix >= thresholds.T)
    predict_score_matrix[negative_index] = 0
    predict_score_matrix[positive_index] = 1
    TP = predict_score_matrix.dot(real_score.T)
    FP = predict_score_matrix.sum(axis=1) - TP
    FN = real_score.sum() - TP
    TN = len(real_score.T) - TP - FP - FN

    fpr = FP / (FP + TN)
    tpr = TP / (TP + FN)
    ROC_dot_matrix = np.mat(sorted(np.column_stack((fpr, tpr)).tolist())).T
    ROC_dot_matrix.T[0] = [0, 0]
    ROC_dot_matrix = np.c_[ROC_dot_matrix, [1, 1]]
    x_ROC = ROC_dot_matrix[0].T
    y_ROC = ROC_dot_matrix[1].T
    auc = 0.5 * (x_ROC[1:] - x_ROC[:-1]).T * (y_ROC[:-1] + y_ROC[1:])

    recall_list = tpr
    precision_list = TP / (TP + FP)
    PR_dot_matrix = np.mat(
        sorted(np.column_stack((recall_list, precision_list)).tolist())
    ).T
    PR_dot_matrix.T[0] = [0, 1]
    PR_dot_matrix = np.c_[PR_dot_matrix, [1, 0]]
    x_PR = PR_dot_matrix[0].T
    y_PR = PR_dot_matrix[1].T
    aupr = 0.5 * (x_PR[1:] - x_PR[:-1]).T * (y_PR[:-1] + y_PR[1:])

    f1_score_list = 2 * TP / (len(real_score.T) + TP - TN)
    accuracy_list = (TP + TN) / len(real_score.T)
    specificity_list = TN / (TN + FP)

    max_index = np.argmax(f1_score_list)
    f1_score = f1_score_list[max_index]
    accuracy = accuracy_list[max_index]
    specificity = specificity_list[max_index]
    recall = recall_list[max_index]
    precision = precision_list[max_index]
    return [aupr[0, 0], auc[0, 0], f1_score, accuracy, recall, specificity, precision]

In [35]:
PATH_ROOT = "/Users/sijianfan/projects/BiSSGL"
PATH_DATA = os.path.join(PATH_ROOT, "datasets/realAnalysis/cdataset/cv_data")

PATH_OUTPUT = os.path.join(PATH_ROOT, "outputs/results/realAnalysis/cdataset/noisy")
if not os.path.isdir(PATH_OUTPUT):
    os.makedirs(PATH_OUTPUT)

PATH_ARCHIVE = os.path.join(PATH_OUTPUT, "archived")
if not os.path.isdir(PATH_ARCHIVE):
    os.makedirs(PATH_ARCHIVE)

In [36]:
filename_staged = os.path.join(PATH_DATA, "staged_dataset.gz")

filenames = {"input": "staged_dataset.gz", "output": "results_bissgl.gz"}

In [37]:
filename_input = os.path.join(PATH_DATA, filenames["input"])

filename_output = os.path.join(PATH_OUTPUT, filenames["output"])

In [51]:
from sgimc.utils import load, save

U, V, Y = load(filename_input)

Y = Y.astype(float)

Y[Y == -1] = 0

In [39]:
import sys

sys.path.append(PATH_ROOT)

from scripts.methods.BiSSGL.BiSSGLc import BiSSGL

In [40]:
U = np.hstack((np.eye(Y.shape[0]), U.toarray()))
V = np.hstack((np.eye(Y.shape[1]), V.toarray()))

In [41]:
from scipy.special import expit
from sgimc.utils import get_submatrix

In [ ]:
model = BiSSGL(
    Y=Y,
    U=U,
    V=V,
    xi=20,
    eta=5e-8,
    tilde_lambda0=10,
    tilde_lambda1=1,
    tilde_alpha=1 / 200 / 5000,
    tilde_beta=1,
    lambda0=10,
    lambda1=1,
    alpha=1 / 200 / 1000,
    beta=1,
    K=200,
    max_iter=5000,
    tol=1e-10,
)

# fit on the whole development dataset
est_mu, est_A, est_B, logLik = model.optimization()

Finished with iterations of 4999


In [ ]:
import numpy as np
from scipy.special import expit

# predicted probability matrix
prob_full = expit(U @ est_A @ est_B.T @ V.T)

# mask for Y == 0
zero_mask = Y == 0

# extract probabilities where Y == 0
zero_probs = prob_full[zero_mask]

# get indices (row, col) of those entries
zero_indices = np.argwhere(zero_mask)

# sort descending by probability
top_k = 10
top_order = np.argsort(-zero_probs)[:top_k]

top_probs = zero_probs[top_order]
top_positions = zero_indices[top_order]

# report
for i in range(top_k):
    r, c = top_positions[i]
    print(f"Rank {i+1}: index=({r}, {c}), predicted_prob={top_probs[i]:.6f}")

Rank 1: index=(265, 38), predicted_prob=0.985958
Rank 2: index=(467, 104), predicted_prob=0.983497
Rank 3: index=(459, 104), predicted_prob=0.980030
Rank 4: index=(238, 104), predicted_prob=0.974086
Rank 5: index=(292, 377), predicted_prob=0.968027
Rank 6: index=(292, 380), predicted_prob=0.968027
Rank 7: index=(436, 38), predicted_prob=0.954201
Rank 8: index=(584, 100), predicted_prob=0.949681
Rank 9: index=(384, 38), predicted_prob=0.942539
Rank 10: index=(101, 380), predicted_prob=0.942108


In [ ]:
import numpy as np
from scipy.special import expit

# predicted probability matrix
prob_full = expit(U @ est_A @ est_B.T @ V.T)

# mask where Y == 0
zero_mask = Y == 0

# probabilities at Y == 0
zero_probs = prob_full[zero_mask]

# positions (row, col) where Y == 0
zero_indices = np.argwhere(zero_mask)

# top 10 highest predicted probabilities
top_k = 10
top_order = np.argsort(-zero_probs)[:top_k]

top_probs = zero_probs[top_order]
top_positions = zero_indices[top_order]

# report using names
for i in range(top_k):
    r, c = top_positions[i]
    drug_name = drugs_features.index[r]
    disease_name = diseases_features.index[c]

    print(
        f"Rank {i+1}: "
        f"Drug = {drug_name}, "
        f"Disease = {disease_name}, "
        f"Predicted probability = {top_probs[i]:.6f}"
    )

Rank 1: Drug = DB00620, Disease = D125600, Predicted probability = 0.985958
Rank 2: Drug = DB01008, Disease = D151380, Predicted probability = 0.983497
Rank 3: Drug = DB00997, Disease = D151380, Predicted probability = 0.980030
Rank 4: Drug = DB00563, Disease = D151380, Predicted probability = 0.974086
Rank 5: Drug = DB00669, Disease = D607498, Predicted probability = 0.968027
Rank 6: Drug = DB00669, Disease = D607508, Predicted probability = 0.968027
Rank 7: Drug = DB00959, Disease = D125600, Predicted probability = 0.954201
Rank 8: Drug = DB01234, Disease = D147540, Predicted probability = 0.949681
Rank 9: Drug = DB00860, Disease = D125600, Predicted probability = 0.942539
Rank 10: Drug = DB00320, Disease = D607508, Predicted probability = 0.942108
